# Meta-Harness: teach an agent to repair better

**What it does.** Meta-Harness improves the program around a fixed model: the
harness that controls its work and context.

**How it works.** A proposer inspects previous harness code, scores and execution
traces, writes a revised harness, and evaluates it on tasks. The
[paper](https://arxiv.org/html/2603.28052v1#S3) searches executable harness programs
and returns a Pareto frontier.

**In this example.** A Claude repair worker edits Python. A separate agent reads
development evidence and revises only its planner instructions or access to the
repository contract. Greedy search, private selection and a held-out comparison
measure those changes through fresh repairs and independent checks.

**Experimental/live-only.** Offline rehearsals verify execution and accounting;
saved live model outputs are still pending. See the
[verification record](https://sentient-xyz.github.io/meta-evolve-docs/project/meta-harness-live-verification/).

| Role | Here |
|---|---|
| Artifact | The worker's planner text and context setting |
| Proposer | `revise`: diagnose development evidence and return a typed harness edit |
| Evaluator | `evaluate`: run the worker, then independently check its repaired code |
| Search | Greedy tries two revisions; private checks select the final harness |
| Result | Harness and repair diffs, phase histories, receipts, and held-out comparison |

**The harness is the candidate.** A repair is the worker's output used to judge
that candidate. The worker model, tests and budgets remain fixed.

**Repair → test → revise the harness → repair again.**

Seven cells take you from a starting harness to an independently checked selection.
Both agents use the Claude Agent SDK. This small study uses four authored repository
contracts; its results demonstrate the mechanism, not benchmark gains.

[Download notebook](https://sentient-xyz.github.io/meta-evolve-docs/downloads/meta-harness-repair.ipynb)
· [Complete source](https://sentient-xyz.github.io/meta-evolve-docs/downloads/meta-harness-example.zip)
· [SDK helper](https://github.com/sentient-xyz/meta-evolve/blob/main/examples/research/meta_harness/repair_agents.py)

<a id="the-concrete-problem"></a>
<a id="follow-the-two-levels"></a>
<a id="setup-and-run"></a>

## 1. Prepare a fresh workspace

Use a **local Jupyter kernel with Python 3.12+** on macOS or Linux. Authenticate
with `claude auth login` in a terminal, or supply `ANTHROPIC_API_KEY` to the kernel's
environment. Run the cells below in order; the same cells can be copied from this
page into an empty notebook. No checkout, earlier lesson, or companion-file upload
is needed. Setup downloads the matching library and verifies the support archive.

The launch check must pass before any agent call. It needs working `sandbox-exec`
on macOS or Landlock ABI 3+ on Linux; hosted notebook environments are unverified.
If an enclosing sandbox prevents launch, use a supported local terminal/kernel.

In [ ]:
%pip install -q https://sentient-xyz.github.io/meta-evolve-docs/downloads/meta-evolve.zip claude-agent-sdk==0.2.156 pydantic==2.12.5
from hashlib import sha256
from io import BytesIO
from pathlib import Path
import sys, tempfile
from urllib.request import urlopen
from zipfile import ZipFile

source_url = "https://sentient-xyz.github.io/meta-evolve-docs/downloads/meta-harness/0a2537d8f2f64adc6a80bab66116bd2471bfa3015b97ac0b4953112d93019947/example.zip"
archive = urlopen(source_url, timeout=30).read()
if sha256(archive).hexdigest() != "0a2537d8f2f64adc6a80bab66116bd2471bfa3015b97ac0b4953112d93019947":
    raise RuntimeError("Example download does not match this notebook.")
support = Path(tempfile.mkdtemp(prefix="meta-harness-source-"))
with ZipFile(BytesIO(archive)) as bundle:
    bundle.extractall(support)
sys.path.insert(0, str(support / "examples/research/meta_harness"))

from typing import Literal

from IPython.display import HTML, display
from pydantic import BaseModel, Field, StrictBool
import meta_evolve as meta
from meta_evolve.domain import ComponentRef
from meta_evolve.harness import HarnessArtifact, PolicyParam, TextParam

from repair_display import final_report, history, results
from repair_evaluation import evaluate_repairs, propose_revision, repair
from repair_execution import preflight
from repair_phases import confirm, save_report

root = Path(tempfile.mkdtemp(prefix="meta-harness-run-"))
preflight(root)
print("Run folder:", root)

<a id="component-worksheet"></a>
<a id="declare-allowed-changes"></a>

## 2. Components of the harness

The starting worker sees the issue and `solution.py`. Setting `include_spec=True`
also gives it the repository's `contract.md`. Planner text changes how it approaches
the repair. Both are typed mutation surfaces; model, tools, tests and budgets stay
outside the candidate. Each worker gets a fresh SDK session with Read, Edit and
Write restricted to its files. The proposer can read its supplied evidence only.

The development issues concern **ledger amounts** and **customer labels**. For
example, the ledger contract defines parentheses as a negative amount and an em
dash as zero. A context change makes that contract available; the worker must still
write working code, which the independent checks measure.

In [ ]:
WORKER_MODEL, PROPOSER_MODEL = "haiku", "sonnet"
seed = HarnessArtifact(
    planner=TextParam(
        "planner", ComponentRef(role="planner", name="repair-worker", version="1"),
        "repair-instructions/v1",
        "Make a small, conservative repair. Preserve the public interface.",
    ),
    context_policy=PolicyParam(
        "context_policy",
        ComponentRef(role="context-policy", name="repository-contract", version="1"),
        "repository-contract/v1", {"include_spec": False},
    ),
)

class HarnessEdit(BaseModel):
    model_config = {"extra": "forbid"}
    surface: Literal["planner", "context_policy"]
    planner: str = Field(max_length=2000, description="New instructions, or empty for a context edit")
    include_spec: StrictBool = Field(description="Whether the worker receives the repository contract")
    diagnosis: str = Field(min_length=1, max_length=1000)
    prediction: str = Field(min_length=1, max_length=1000)

print("Worker:", WORKER_MODEL, "· proposer:", PROPOSER_MODEL)
print("Starting instructions:", seed.planner.value)
print("Repository contract available:", seed.context_policy.configuration["include_spec"])

## 3. Measure the worker's actual repairs

`repair` asks the SDK worker to edit a real file, captures it, and runs it in a
confined Python process. The evaluator compares its answers with fixed expectations
in the parent process. Each task has four checks; quality is the fraction passed.
The table appears as each harness is evaluated in cell 5.

The worker receives neither the hidden expected answers nor later tasks. A completed
wrong answer or task exception fails a check. Invalid Python, timeouts, SDK errors
and infrastructure failures are retained as unrankable outcomes. Unknown SDK usage
stops the run and keeps its receipt.

In [ ]:
def evaluate(harness):
    measured = evaluate_repairs(
        harness, split="development", model=WORKER_MODEL, root=root, worker=repair,
    )
    display(HTML(results(measured)))
    return measured

<a id="how-a-cited-edit-becomes-a-candidate"></a>

## 4. Let an agent diagnose one change

The selected development evidence contains the parent harness, its repaired source,
pass/fail checks, and observable SDK traces. The proposer reads those files through
the SDK and returns a structured `HarnessEdit`. The helper validates the declared
surface, preserves the diagnosis and prediction, and cites the supplied evidence.
Private and held-out observations never enter this workspace.

In [ ]:
def revise(harness, *, context):
    feedback = context.evidence.latest("meta-harness.development")
    return propose_revision(
        harness, feedback, model=PROPOSER_MODEL, root=root, schema=HarnessEdit,
    )

<a id="the-actual-development-declaration"></a>

## 5. Try two revisions

This cell evaluates the seed and makes at most two proposed revisions. Greedy
selection retains the highest development score, favoring the earlier occurrence
on a tie. Both attempts stay visible even when they fail or do not improve quality.

**Live calls begin here.** Run All makes at most **13 SDK calls** across all phases.
Each session allows eight turns and $0.35 of SDK-estimated cost. Allow several
minutes; progress and check tables appear as calls finish. Billed spend is unknown.

In [ ]:
run = meta.run(
    task=meta.Task(evaluator=evaluate, artifact=HarnessArtifact,
        objectives=(meta.Maximize("quality"),),
        budget=meta.Budget(trials=2, evaluations=3, tokens=400_000, wall_seconds=1800)),
    seed=seed, proposer=revise, search=meta.Greedy(max_trials=2),
    context=meta.RecentAncestors(max_records=12, max_chars=24000), random_seed=0,
    storage=meta.Storage.durable(root / "development"),
)

<a id="inspect-the-repair"></a>

## 6. Follow the evidence

Read the harness change, then expand a repair to see the actual code diff and its
checks. Worker traces are available beneath each repair. This cell reads retained
results and makes no model calls; it is safe to rerun.

In [ ]:
display(HTML(history(run)))
print("Search:", run.usage().trials, "proposals ·", run.usage().evaluations, "evaluations")
print("Run folder:", root)

<a id="check-fresh-repairs"></a>
<a id="export-and-adopt"></a>

## 7. Check fresh tasks and keep the result

Every rankable development occurrence receives the same private **warehouse counts**
task. The highest score wins; ties favor the earliest occurrence. This can overturn
the development winner. The selection is written to disk before the untouched
**shipping priorities** task runs once for the original seed and once for the
selected harness, with identical model and limits.

Only a strict held-out improvement exports `selected-harness`. Ties, regressions,
failures, and a selection that keeps the seed are valid results. The report includes
usage from every phase and reconciles tokens against the saved SDK receipts.

In [ ]:
phases = confirm(run, seed=seed, model=WORKER_MODEL, root=root)
report = save_report(run, phases, root)
display(HTML(final_report(report)))
print("Report:", root / "report.json")
print("Export:", root / report["exported"] if report["exported"] else "No eligible export")

<a id="change-and-predict"></a>

Keep the printed run folder: it contains `report.json`, `selection.json`, all SDK
receipts and durable phase histories. These local SDK callables do not support
resume. For another experiment, restart the kernel and run from setup to create
a new folder; rerunning a paid phase against an existing store is not a fresh run.
To try your own agent, change the two model settings or starting instructions
before cell 5. Keep them fixed through all phases of each comparison.

<a id="what-the-paper-does"></a>

This is a bounded live adaptation of
[Meta-Harness, Algorithm 1](https://arxiv.org/html/2603.28052v1#S3): agent diagnosis
uses prior harness behavior, followed by independent evaluation. The paper searches
executable harness programs and returns a Pareto frontier. This lesson changes
planner/context settings under Greedy search and adds private selection.

<a id="complete-supporting-source"></a>
<a id="evidence-and-ownership"></a>

The [deterministic reference](https://sentient-xyz.github.io/meta-evolve-docs/research/meta-harness-reference/) retains the
full M8 audit, confinement, source-citation and replay study. The
[introductory help assistant](https://sentient-xyz.github.io/meta-evolve-docs/guides/harness-evolution/) is a credential-free
first step. See the [harness contract](https://sentient-xyz.github.io/meta-evolve-docs/architecture/#programming-harnesses)
and [Claude SDK reference](https://code.claude.com/docs/en/agent-sdk/python).